# How governments buy AI: rerun the study on live data

This notebook reruns the Sell to State study [How governments buy AI](https://www.selltostate.com/blog/how-governments-buy-ai/) on the current Sell to State data, through the Sell to State API.

It finds every public tender whose **title** names AI, labels it with the study's rules (what the tender buys, what the AI does, who it serves, which brands it names), and builds the study's main tables and charts: AI tenders per 10,000 records by country and year, what governments buy, prices in US dollars, and the AI tenders that are open now.

**You need access to the Sell to State API**, one of:
- an API key from [app.selltostate.com](https://app.selltostate.com) (Settings > API), as `STS_API_KEY`;
- an agent client: the first cell signs up through `POST /api/oidc/register` and prints a payment link ($49 a month). After the payment, the client id works as `STS_CLIENT_ID`.

A full run on all countries makes about 6,500 API calls and takes about 25 minutes. The data is saved in `data/`, so a rerun continues where it stopped.

**How the results differ from the article.** The article used the original sources (TED, the Open Contracting Data Registry, USAspending, SECOP II, Prozorro, p-portal, CanadaBuys) and also read the descriptions of the tenders. The API searches titles, buyer names and supplier names. So this notebook counts the AI that a title names: in the article that was 55% of all AI procurements. Sell to State also covers a different set of countries and years, and its coverage grew in 2026, so compare a country with itself inside one year first.

In [ ]:
import os, getpass
import pandas as pd
import matplotlib.pyplot as plt
from aig.client import STS, register
from aig import study, geo

if not (os.environ.get('STS_API_KEY') or os.environ.get('STS_CLIENT_ID')):
    key = getpass.getpass('Sell to State API key (press Enter to sign up as an agent): ').strip()
    if key:
        os.environ['STS_API_KEY'] = key
    else:
        reg = register()                        # agent signup: no data access until the payment
        print('Pay here, then come back:', reg['payment_link'])
        input('Press Enter after the payment.')
        os.environ['STS_CLIENT_ID'] = reg['client_id']   # keep it private: it is the credential
sts = STS()
sts.whoami()                                    # "active": true means the data calls will work

## 1. Countries
Sell to State covers these countries. Keep all of them, or set `COUNTRIES` to a short list for a quick run.

In [ ]:
all_countries = study.countries(sts)
COUNTRIES = os.environ.get('AIG_COUNTRIES', '').split(',') if os.environ.get('AIG_COUNTRIES') else sorted(all_countries)
YEARS = (2024, 2025, 2026)
pd.Series(all_countries, name='records').sort_values(ascending=False).to_frame().head(70)

## 2. Collect AI candidates and denominators
Each country is searched with the English terms and the terms of its languages (`aig/terms.py`). The search is broad: "AI" also finds "Air". The keyword rules decide in the next step.

In [ ]:
hits = study.collect(sts, COUNTRIES, year_min=min(YEARS), year_max=max(YEARS))
den = study.denominators(sts, YEARS)
print(len(hits), 'search hits;', sts.calls, 'API calls')

## 3. Label the AI tenders
The keyword families (`aig/families.py`) remove false matches in 25 languages: "AI" as a Romanian word, "IA" as a customs office, "Copilot" as an aircraft part. Then the taxonomy labels each tender.

In [ ]:
ai = study.label(hits)
ai.to_csv(os.path.join(study.DATA, 'ai_tenders.csv'), index=False)
print(len(ai), 'AI tenders')
ai[['country', 'year', 'title', 'buyer', 'purchase', 'capability', 'value_usd', 'app_url']].head(20)

## 4. AI tenders per 10,000 records, by country and year
The unit is the Sell to State record. A country needs 2,000 records in the year to appear.

In [ ]:
s = study.shares(ai, den)
s.to_csv(os.path.join(study.DATA, 'shares.csv'), index=False)
study.chart_ranking(s, max(YEARS)); plt.show()
s.pivot_table(index='name', columns='year', values='per_10k').round(1).sort_values(max(YEARS), ascending=False).head(40)

In [ ]:
# median country of each region, by year
s.groupby(['region', 'year']).per_10k.median().unstack().round(1)

## 5. What governments buy, what the AI does, who it serves

In [ ]:
study.chart_mix(study.mix(ai, 'purchase'), 'What the tender buys'); plt.show()
tech = ai[~ai.capability.str.startswith('n/a')]
study.chart_mix(study.mix(tech, 'capability'), 'What the AI does (technology purchases)'); plt.show()
study.mix(ai, 'purchase', by='region')

In [ ]:
# use cases inside each capability, and who the AI serves
for cap in ['Computer vision and imaging', 'Speech and transcription', 'Generative AI and LLMs', 'Prediction, risk and analytics', 'Chatbots and virtual assistants']:
    x = ai[ai.capability == cap]
    if len(x): print(cap, len(x)); print(study.mix(x, 'use_case').to_string(), '\n')
study.mix(ai, 'audience')

In [ ]:
# brands named in the titles, with the median contract in US dollars
study.brands(ai)

## 6. Prices in US dollars
Percentiles of the contract value. Amounts are converted at 2025 average rates (`aig/geo.py`). Framework ceilings can pass as values; check the large rows by hand.

In [ ]:
for by in ['region', 'country', 'purchase']:
    q = study.prices(ai, by)
    study.chart_prices(q.rename(index=lambda i: geo.NAME.get(i, i)), f'AI contract values by {by}'); plt.show()
study.prices(ai, 'purchase').round(0)

In [ ]:
# median price of each purchase type in each region (cells with 10 or more contracts)
v = ai.dropna(subset=['value_usd'])
m = v.groupby(['purchase', 'region']).value_usd.agg(['median', 'count'])
m[m['count'] >= 10]['median'].unstack().round(-2)

## 7. AI tenders that are open now
The same search with `lifecycle=open`. Each row links to the tender in the Sell to State app.

In [ ]:
open_hits = study.collect(sts, COUNTRIES, year_min=2025, year_max=2027, lifecycle='open', name='open_hits.jsonl')
open_ai = study.label(open_hits)
open_ai.to_csv(os.path.join(study.DATA, 'open_ai_tenders.csv'), index=False)
print(len(open_ai), 'open AI tenders')
open_ai.sort_values('value_usd', ascending=False)[['country', 'title', 'buyer', 'purchase', 'capability', 'value_usd', 'app_url']].head(40)

## 8. Cite and share
The method, the rules and the article: [How governments buy AI](https://www.selltostate.com/blog/how-governments-buy-ai/), Sell to State. The code is MIT-licensed. The data belongs to the public sources that Sell to State collects; follow the API terms of use when you publish rows.